# Scraping Commentaires Facebook — v2
**Updates:** Integrated Universal Facebook URL Normalizer. > All formats (/posts/, /photo/?fbid=, /videos/, /reel/, /permalink/) are automatically converted to the /id_account/posts/id_post structure before scraping. This ensures full access to all comments.

## 1. Setup & Driver

In [1]:
import undetected_chromedriver as uc
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time
import re
import pickle
import pandas as pd
from urllib.parse import urlparse, parse_qs

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-automation")

driver = uc.Chrome(options=options)
driver.get('https://www.google.com')
time.sleep(4)
# --> Log in manually then come back here

## 2. Universal Facebook URL Normalizer

This is the core new function. It detects the URL type and converts any Facebook URL format into the canonical `/id_account/posts/id_post` form that gives full comment access.

| Input format | Strategy |
|---|---|
| `/posts/` | Already correct — pass through |
| `/photo/?fbid=` | Load page, extract `owner_id` + `story_fbid` from HTML |
| `/videos/` | Load page, extract owner + post IDs from HTML |
| `/reel/` | Load page, extract owner + post IDs from HTML |
| `/permalink/` | Follow redirect, then re-detect |
| Unknown | Return as-is with warning |

In [2]:
def extract_ids_from_page_source(driver):
    """
    Scrapes the current page source to find owner_id and post/story_fbid.
    Facebook embeds these in the page's JSON-like data blobs.
    Returns (owner_id, post_id) or (None, None) if not found.
    """
    html = driver.page_source

    # Pattern priority list — from most specific to most generic
    owner_patterns = [
        r'"page_id"\s*:\s*"?(\d+)"?',
        r'"owner_id"\s*:\s*"?(\d+)"?',
        r'"owner"\s*:\s*\{[^}]*"id"\s*:\s*"?(\d+)"?',
        r'"profileID"\s*:\s*"?(\d+)"?',
        r'"actorID"\s*:\s*"?(\d+)"?',
    ]

    post_patterns = [
        r'"story_fbid"\s*:\s*"?(\d+)"?',
        r'"post_id"\s*:\s*"?(\d+)"?',
        r'"toppableFeedID"\s*:\s*"?(\d+)"?',
        r'"content_id"\s*:\s*"?(\d+)"?',
    ]

    owner_id = None
    for pat in owner_patterns:
        m = re.search(pat, html)
        if m:
            owner_id = m.group(1)
            break

    post_id = None
    for pat in post_patterns:
        m = re.search(pat, html)
        if m:
            post_id = m.group(1)
            break

    return owner_id, post_id


def normalize_fb_url(url, driver, verbose=True):
    """
    Universal Facebook URL normalizer.
    Converts any FB URL format to https://www.facebook.com/{owner_id}/posts/{post_id}
    so that the full comment section is accessible.

    Args:
        url     : The original Facebook URL (any format)
        driver  : The active Selenium WebDriver (must be logged in)
        verbose : Print resolution steps if True

    Returns:
        Normalized URL string
    """
    parsed = urlparse(url)
    path = parsed.path

    # ── Case 1: Already in /id/posts/id format ──────────────────────────────
    if re.match(r'/\d+/posts/\d+', path):
        if verbose:
            print(f"  [URL] Already normalized: {url}")
        return url

    # ── Case 2: Named page /pagename/posts/id ───────────────────────────────
    if re.match(r'/[^/]+/posts/\d+', path):
        if verbose:
            print(f"  [URL] Named-page /posts/ URL — loading to resolve owner ID...")
        driver.get(url)
        time.sleep(4)
        owner_id, post_id = extract_ids_from_page_source(driver)
        if owner_id and post_id:
            normalized = f"https://www.facebook.com/{owner_id}/posts/{post_id}"
            if verbose:
                print(f"  [URL] Resolved to: {normalized}")
            return normalized
        # fallback: keep as-is (named page posts still work)
        return url

    # ── Case 3: /photo/?fbid=XXXX ───────────────────────────────────────────
    if '/photo/' in path or '/photo' in path:
        fbid = parse_qs(parsed.query).get('fbid', [None])[0]
        if verbose:
            print(f"  [URL] Photo URL detected (fbid={fbid}) — loading to resolve...")
        driver.get(url)
        time.sleep(4)
        owner_id, post_id = extract_ids_from_page_source(driver)
        if owner_id and post_id:
            normalized = f"https://www.facebook.com/{owner_id}/posts/{post_id}"
            if verbose:
                print(f"  [URL] Resolved to: {normalized}")
            return normalized
        # Fallback: use fbid as post_id (may limit comments but better than nothing)
        if verbose:
            print(f"  [URL] ⚠ Could not extract IDs from page. Falling back to original URL.")
        return url

    # ── Case 4: /videos/XXXX ────────────────────────────────────────────────
    if '/videos/' in path:
        if verbose:
            print(f"  [URL] Video URL detected — loading to resolve...")
        driver.get(url)
        time.sleep(4)
        owner_id, post_id = extract_ids_from_page_source(driver)
        if owner_id and post_id:
            normalized = f"https://www.facebook.com/{owner_id}/posts/{post_id}"
            if verbose:
                print(f"  [URL] Resolved to: {normalized}")
            return normalized
        return url

    # ── Case 5: /reel/XXXX ──────────────────────────────────────────────────
    if '/reel/' in path:
        if verbose:
            print(f"  [URL] Reel URL detected — loading to resolve...")
        driver.get(url)
        time.sleep(4)
        owner_id, post_id = extract_ids_from_page_source(driver)
        if owner_id and post_id:
            normalized = f"https://www.facebook.com/{owner_id}/posts/{post_id}"
            if verbose:
                print(f"  [URL] Resolved to: {normalized}")
            return normalized
        return url

    # ── Case 6: /permalink/ ─────────────────────────────────────────────────
    if '/permalink/' in path:
        if verbose:
            print(f"  [URL] Permalink detected — following redirect...")
        driver.get(url)
        time.sleep(4)
        resolved_url = driver.current_url
        if resolved_url != url:
            return normalize_fb_url(resolved_url, driver, verbose)  # recurse
        return url

    # ── Fallback ─────────────────────────────────────────────────────────────
    if verbose:
        print(f"  [URL] ⚠ Unknown URL format — using as-is: {url}")
    return url


print("URL normalizer loaded.")

URL normalizer loaded.


## 3. Facebook Comment Scraper (unchanged from v1)

In [3]:
def scrape_facebook_comments(driver):
    comments_data = []

    top_level_comments = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((
            By.XPATH,
            "//div[@aria-label='Comment' or @role='article']"
        ))
    )

    print(f"Found {len(top_level_comments)} top-level comment containers")

    for c in top_level_comments:
        try:
            comment_divs = c.find_elements(By.XPATH, ".//div[@dir='auto']")
            full_comment = "\n".join([d.text for d in comment_divs if d.text.strip() != ""])
            if full_comment:
                comments_data.append({"comment": full_comment})
        except:
            continue

    return comments_data


print("Comment scraper loaded.")

Comment scraper loaded.


## 4. Main Scraping Loop — Universal (all Facebook URL types)

This loop works for **any** Facebook link format in `links_to_scrape`.  
Just feed it your list of links — `/posts/`, `/photo/`, `/videos/`, `/reel/`, etc. — and the normalizer handles the rest.

In [4]:
## Load your Excel data (adjust sheet name if needed)
df = pd.read_csv("Facebook_Links_example.csv")
df.head(2)

,Facebook Link
0,https://www.facebook.com/photo/?fbid=122173550...
1,https://www.facebook.com/photo/?fbid=122173435...


In [5]:
## Collect ALL links regardless of type into one unified list
## The normalizer will handle /photo/, /posts/, /videos/, /reel/ etc. automatically

all_fb_links = df['Facebook Link'].dropna().tolist()

print(f"Total Facebook links to process: {len(all_fb_links)}")

# Quick breakdown by detected type
for link in all_fb_links[:5]:
    print(" ", link)

Total Facebook links to process: 5
  https://www.facebook.com/photo/?fbid=122173550774906522
  https://www.facebook.com/photo/?fbid=122173435634906522
  https://www.facebook.com/photo/?fbid=122172850226906522
  https://www.facebook.com/reel/798646192704615
  https://www.facebook.com/photo/?fbid=122253938882179483


In [6]:
driver.get('https://www.facebook.com')

In [12]:
df_comms_facebook = []
failed_urls = []  # Track any links that couldn't be resolved

for original_url in all_fb_links:
    print(f"\n{'='*60}")
    print(f"Original URL: {original_url}")

    # ── STEP 1: Normalize the URL ──────────────────────────────────────────
    try:
        web = normalize_fb_url(original_url, driver, verbose=True)
    except Exception as e:
        print(f"Normalization failed: {e}")
        failed_urls.append({"url": original_url, "reason": f"normalize error: {e}"})
        continue

    # ── STEP 2: Navigate (only if normalizer didn't already load the page) ─
    if driver.current_url.rstrip('/') != web.rstrip('/'):
        driver.get(web)
        time.sleep(5)
    else:
        # Page already loaded during normalization — just wait a bit more
        time.sleep(2)

    print(f"  Scraping: {web}")

    # ── STEP 3: Select 'Tous les commentaires' ────────────────────────────
    try:
        sort_dropdown_present = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//div[@role='button'][descendant::span[contains(text(), 'Plus pertinents')]]"
            ))
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", sort_dropdown_present)
        time.sleep(2)

        sort_dropdown_clickable = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//div[@role='button'][descendant::span[contains(text(), 'Plus pertinents')]]"
            ))
        )
        driver.execute_script("arguments[0].click();", sort_dropdown_clickable)

        all_comments_option = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//div[@role='menuitem']//span[contains(text(), 'Tous les commentaires')]"
            ))
        )
        driver.execute_script("arguments[0].click();", all_comments_option)
        print("  Switched to 'All comments'.")

    except Exception as e:
        print(" No comment dropdown found — post probably has 0 comments. Skipping…")
        continue

    # ── STEP 4: Scroll to bottom ──────────────────────────────────────────
    try:
        scroll_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//div[@data-visualcompletion='ignore' and contains(@style, 'height')]"
            ))
        )
        scroll_box = scroll_container.find_element(By.XPATH, "./..")
        last_height = 0

        while True:
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", scroll_box)
            time.sleep(2)
            new_height = driver.execute_script("return arguments[0].scrollHeight;", scroll_box)
            if new_height == last_height:
                break
            last_height = new_height

    except Exception as e:
        print(f"    Could not scroll — skipping. ({e})")
        continue

    # ── STEP 5: Scrape comments ───────────────────────────────────────────
    all_comments = scrape_facebook_comments(driver)
    print(f"  Total clean comments scraped: {len(all_comments)}")

    if len(all_comments) <= 1:
        print("  No comments extracted — skipping.")
        continue

    df = pd.DataFrame(all_comments[1:])
    df['comment'] = df['comment'].str.replace('\n', '', regex=False)
    df['Origin_of_comms'] = original_url   # Keep original URL for traceability
    df['Normalized_URL'] = web             # Also store the resolved URL

    df_comms_facebook.append(df)
    time.sleep(1)

print(f"\n Done. Total posts scraped: {len(df_comms_facebook)}")
if failed_urls:
    print(f"{len(failed_urls)} URLs failed:")
    for f in failed_urls:
        print(f"   {f}")


Original URL: https://www.facebook.com/photo/?fbid=122173550774906522
  [URL] Photo URL detected (fbid=122173550774906522) — loading to resolve...
  [URL] Resolved to: https://www.facebook.com/61577195680838/posts/122173550804906522
  Scraping: https://www.facebook.com/61577195680838/posts/122173550804906522
  Switched to 'All comments'.
Found 26 top-level comment containers
  Total clean comments scraped: 21

Original URL: https://www.facebook.com/photo/?fbid=122173435634906522
  [URL] Photo URL detected (fbid=122173435634906522) — loading to resolve...
  [URL] Resolved to: https://www.facebook.com/61577195680838/posts/122173435670906522
  Scraping: https://www.facebook.com/61577195680838/posts/122173435670906522
  Switched to 'All comments'.
Found 22 top-level comment containers
  Total clean comments scraped: 14

Original URL: https://www.facebook.com/photo/?fbid=122172850226906522
  [URL] Photo URL detected (fbid=122172850226906522) — loading to resolve...
  [URL] Resolved to: htt

## 5. Save Results

In [13]:
if df_comms_facebook:
    final_df = pd.concat(df_comms_facebook, ignore_index=True)
    print(f"Total comments collected: {len(final_df)}")
    print(f"From {final_df['Origin_of_comms'].nunique()} unique posts")
    display(final_df.head())

    final_df.to_csv(
        "facebook_comments_v2.csv",
        index=False,
        sep="|",
        encoding="cp1252",
        errors="replace"
    )
    print("Saved to facebook_comments_v2.csv")
else:
    print("No comments were collected.")

Total comments collected: 167
From 4 unique posts


,comment,Origin_of_comms,Normalized_URL
0,You are late.Key points by Mojtaba Khamenei. S...,https://www.facebook.com/photo/?fbid=122173550...,https://www.facebook.com/61577195680838/posts/...
1,The most detailed targets that should be set u...,https://www.facebook.com/photo/?fbid=122173550...,https://www.facebook.com/61577195680838/posts/...
2,Mashallahlive long Syed Ayottullah Ali Mujtaba...,https://www.facebook.com/photo/?fbid=122173550...,https://www.facebook.com/61577195680838/posts/...
3,"Iran Today perlu diwaspadai....., ketika ameri...",https://www.facebook.com/photo/?fbid=122173550...,https://www.facebook.com/61577195680838/posts/...
4,We lovesri lanka...,https://www.facebook.com/photo/?fbid=122173550...,https://www.facebook.com/61577195680838/posts/...


Saved to facebook_comments_v2.csv
